In [1]:
# NOTEBOOK 42 - real explainability and real uncertainty for the frontend, from the FINAL model (dyn_adv, 5 seeds, real day-old forecast wind)
# Notebook 30 did this for the v2 models. This is the same export for the final model, on the same 970 playback windows (so the window indices line up), in a NEW folder:
#   v2_clean/frontend_export_dynadv/
#     explain_windows.json : per window and station - exact Shapley impact (ug/m3) of EIGHT players on the mean 24-hour forecast, the baseline (all players at the training mean),
#                            the forecast, and the real attention over the 48 input hours
#     bands.json           : per window, station and horizon - 10th / 50th / 90th percentile across the 5 seeds (ensemble spread, not a calibrated interval)
#     explain_global.json  : window-averaged impacts and shares from this run
# The eight players: PM2.5 history | Other pollutants | Weather (T, RH) | Wind (input wind + forecast wind) | City context | Calendar (input + future)
#                    | Upwind PM2.5 (the transport feature) | Wind graph (switched off = no neighbour exchange, only the self-loop)
# The last two are the wind-aware parts of the model, so this gives a direct measurement of what they contribute.
# Checks: (1) the ensemble mean reproduces the saved test predictions (dynadv_test_pred_forecast.npy), (2) the window indices line up with the playback windows (actuals),
#         (3) Shapley efficiency (baseline + impacts = forecast).
# Nothing in frontend/ is touched. 256 coalitions x 5 models per window: about 15-30 minutes on the GPU for all 970 windows. LIMIT = a small number tests it quickly.
# HOW TO CHECK: this cell only prints a note.
LIMIT = None          # e.g. 12 for a quick test; None = all playback windows
print('frontend export for the final model; LIMIT =', LIMIT)

frontend export for the final model; LIMIT = None


In [2]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
PREVDAY_NPY     = os.path.join(V2_DIR, 'openmeteo_prevday1_uv.npy')
ADJ_FC_NPY      = os.path.join(V2_DIR, 'dynamic_adj_prevday1.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [3]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')
prevday = np.load(PREVDAY_NPY).astype(np.float32)                    # day-old forecast wind (T,N,2), blowing TOWARD
adj_fc = np.load(ADJ_FC_NPY).astype(np.float32)                      # forecast-wind graph for the future hours (T,N,N)
assert prevday.shape == era5.shape and adj_fc.shape == dyn_adj.shape, (prevday.shape, adj_fc.shape)
print('forecast wind', prevday.shape, '| forecast graph', adj_fc.shape, '| mean |forecast - ERA5| wind speed difference (m/s):',
      float(np.abs(np.hypot(prevday[..., 0], prevday[..., 1]) - era5_spd).mean()))

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508
forecast wind (19921, 37, 2) | forecast graph (19921, 37, 37) | mean |forecast - ERA5| wind speed difference (m/s): 0.7865347862243652


In [4]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

C:\Users\jlakk_j7eh5fd\AppData\Local\Temp\ipykernel_10120\1524811802.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month


F=26  NaNs 0


In [5]:
# 1) geometry and the two new feature pairs for Delhi
lats = df_st['latitude'].values.astype(float); lons = df_st['longitude'].values.astype(float)
THRESHOLD_KM, MAX_LAG = 12.0, 6

def geometry(lats, lons):
    n = len(lats)
    dist = np.zeros((n, n)); du = np.zeros((n, n)); dv = np.zeros((n, n))
    R = 6371.0
    for i in range(n):
        for j in range(n):
            if i != j:
                p1, p2 = np.radians(lats[i]), np.radians(lats[j])
                a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lons[j] - lons[i]) / 2) ** 2
                dist[i, j] = 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
                de = (lons[j] - lons[i]) * np.cos(np.radians(lats[i])); dn = lats[j] - lats[i]; nr = np.hypot(de, dn)
                if nr > 0:
                    du[i, j], dv[i, j] = de / nr, dn / nr          # unit vector from i to j
    return dist, du, dv, (dist > 0) & (dist <= THRESHOLD_KM)

def transport_features(pm, uv, dist, du, dv, within):
    # pm (T,n) raw PM2.5, uv (T,n,2) ERA5 wind (blowing TOWARD). Only values at or before hour t are used (no leakage).
    T, n = pm.shape
    adv = pm.copy(); nb = pm.copy()
    src = np.arange(n)[:, None] * np.ones((1, n), dtype=int)           # source station index j for entry [j, i]
    stat = within.astype(float); cnt = stat.sum(0); has_nb = cnt > 0
    for t in range(T):
        align = uv[t, :, 0][:, None] * du + uv[t, :, 1][:, None] * dv   # wind at j along the direction j -> i  (m/s)
        w = np.where(within & (align > 0), align / (dist + 1e-6), 0.0)  # j is upwind of i
        wsum = w.sum(0); has = wsum > 0
        lag = np.clip(np.rint(dist / (np.maximum(align, 0.5) * 3.6)), 0, MAX_LAG).astype(int)   # hours of travel (km / (km/h))
        vals = pm[np.clip(t - lag, 0, T - 1), src]                      # pm[t - lag[j,i], j]
        adv[t, has] = ((w * vals).sum(0))[has] / wsum[has]
        nb[t, has_nb] = ((stat * pm[t][:, None]).sum(0))[has_nb] / cnt[has_nb]
    return adv, nb

dist, du, dv, within = geometry(lats, lons)
t0 = time.time()
adv, nb = transport_features(X_raw[:, :, 0].astype(float), era5, dist, du, dv, within)
pm0 = X_raw[:, :, 0]
extra = np.stack([adv - 0, adv - pm0, nb - 0, nb - pm0], axis=2).astype(np.float32)       # adv_pm, adv_diff, nb_pm, nb_diff
extra[:, :, 0] = adv; extra[:, :, 2] = nb
X30 = np.concatenate([X_raw, extra], axis=2)
print(f'features built in {time.time()-t0:.0f}s | X30 {X30.shape}')
print('mean |adv - own| :', float(np.abs(adv - pm0).mean()), '| mean |nb - own| :', float(np.abs(nb - pm0).mean()))
print('corr(adv, own PM):', float(np.corrcoef(adv.ravel(), pm0.ravel())[0, 1]), '| corr(nb, own PM):', float(np.corrcoef(nb.ravel(), pm0.ravel())[0, 1]))

# HOW TO CHECK: about a minute. Both correlations are high (0.8-0.95) because neighbouring stations resemble each other;
# the question for the models is whether the lagged, wind-aligned version adds anything beyond the same-hour mean.

features built in 1s | X30 (19921, 37, 30)
mean |adv - own| : 28.104649143919474 | mean |nb - own| : 23.36611597159713
corr(adv, own PM): 0.8577609401778713 | corr(nb, own PM): 0.9041709851392609


In [6]:
# 2) scaler for the 30 channels (fit on the training period only); the first 26 must match the saved v3 scaler
from sklearn.preprocessing import StandardScaler
import pickle
T = X30.shape[0]; tr_end = int(T * 0.70)
sc30 = StandardScaler().fit(X30[:tr_end].reshape(-1, 30))
v3s = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
assert np.allclose(sc30.mean_[:26], v3s.mean_, rtol=1e-4, atol=1e-4) and np.allclose(sc30.scale_[:26], v3s.scale_, rtol=1e-4, atol=1e-4), 'scaler mismatch'
log_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Xs_full = sc30.transform(X30.reshape(-1, 30)).reshape(T, N, 30).astype(np.float32)
Yl_full = to_log(X30[:, :, 0])
Y_raw = X30[:, :, 0].copy()
print('scaler check passed: the first 26 channels match the v3 scaler')

# HOW TO CHECK: prints the pass line. An assertion error means the data or train split changed - stop and tell me.

scaler check passed: the first 26 channels match the v3 scaler


C:\Users\jlakk_j7eh5fd\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
# future-hour wind channels from the day-old forecast (scaled with the same train-period scaler as the observed wind)
pv_spd = np.hypot(prevday[..., 0], prevday[..., 1])
pv_raw = np.stack([prevday[..., 0], prevday[..., 1], pv_spd], axis=2).astype(np.float32)          # (T,N,3)
WIDX = [U_IDX, V_IDX, W_IDX]
Xs_fut = Xs_full.copy()
Xs_fut[:, :, WIDX] = ((pv_raw - sc30.mean_[WIDX]) / sc30.scale_[WIDX]).astype(np.float32)
print('future wind channels now come from the day-old forecast | scaled means: observed', Xs_full[:, :, WIDX].mean((0, 1)).round(2), 'forecast', Xs_fut[:, :, WIDX].mean((0, 1)).round(2))

# HOW TO CHECK: the u and v means are close; the forecast speed mean is lower than the observed one (about -0.5 against -0.1 on the scaled axis) because the day-old forecast runs slower than ERA5. That bias is real and is why the models are retrained on it.

future wind channels now come from the day-old forecast | scaled means: observed [ 0.04  0.01 -0.13] forecast [-0.07  0.14 -0.5 ]


In [8]:
# model with three graph modes plus the learned blend, and optional context dropout at training time
CTX_COLS = ['city_mean_pm', 'city_max_pm', 'pm_minus_city']
CTX_IDX = [FEATURE_COLS.index(c) for c in CTX_COLS]
WSPD_IDX = FEATURE_COLS.index('wspd')
FUT_WSPD = 9                                  # wspd is the last of the 10 future channels (7 calendar + u, v, wspd)

class WATGCNv4(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon, n_stations,
                 dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        sa = sa.clone(); sa.fill_diagonal_(0)
        self.register_buffer('static_adj', sa, persistent=False)
        self.register_buffer('static_rn', sa / sa.sum(1, keepdim=True).clamp(min=1.0), persistent=False)   # row-normalised
        if graph_mode == 'blend':
            self.blend_a = nn.Parameter(torch.zeros(1)); self.blend_b = nn.Parameter(torch.zeros(1))

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def alpha(self, speed):                       # speed: (B, steps) standardised wind speed -> weight of the wind graph
        return torch.sigmoid(self.blend_a + self.blend_b * speed)[..., None, None]

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            A = None
        elif self.graph_mode == 'dynamic':
            A = self._n(At + I.view(1, 1, n, n))
        elif self.graph_mode == 'static':
            A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
        else:                                      # blend
            a = self.alpha(X[..., WSPD_IDX].mean(-1))
            A = self._n(a * At + (1 - a) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n))
        for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
            h = self.act(nrm(lin(h if A is None else torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        elif self.graph_mode == 'static':
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        else:
            af = self.alpha(Fut[..., FUT_WSPD].mean(-1))
            ctx_fc = torch.matmul(self._n(af * Afc + (1 - af) * self.static_rn.view(1, 1, n, n) + I.view(1, 1, n, n)), ctx_rep)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, self.fut_enc(Fut)], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

In [9]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

In [10]:
# window set, dataset (forecast setting: day-old forecast wind and forecast graph), the 5 final models
SEEDS = [42, 7, 123, 2024, 999]
ADV_COLS = list(range(26)) + [26, 27]
ADV_NAMES = FEATURE_COLS + ['adv_pm', 'adv_diff']
STATIC_NP = np.load(STATIC_ADJ_NPY).astype(np.float32)
val_end = int(T * 0.85)
def split_starts(lo, hi): return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
TEST_STARTS = split_starts(val_end, T)
class DSE(Dataset):
    def __init__(self, starts, Xfut, Afc): self.s, self.Xfut, self.Afc = np.asarray(starts, np.int64), Xfut, Afc
    def __len__(self): return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(np.ascontiguousarray(Xs_full[g:i1][:, :, ADV_COLS])), torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(self.Afc[i1:i2].copy()), torch.from_numpy(np.ascontiguousarray(self.Xfut[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1 - 1].copy()), torch.from_numpy(Yl_full[i1:i2].copy()), torch.tensor(1.0))
ds = DSE(TEST_STARTS, Xs_fut, adj_fc)

PB = json.load(open(os.path.join(V2_DIR, 'dashboard_export', 'playback.json')))
widx = [int(f['window_idx']) for f in PB]
if LIMIT: widx = widx[:LIMIT]; PB = PB[:LIMIT]
assert len(TEST_STARTS) == 2918 and max(widx) < len(TEST_STARTS), (len(TEST_STARTS), max(widx))

models = []
for sd in SEEDS:
    m = WATGCNv4(len(ADV_COLS), HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT, N_FUT, GCN_HOPS, graph_mode='dynamic', static_adj=STATIC_NP).to(DEVICE)
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v5dyn_adv_seed{sd}.pt'), map_location=DEVICE)); m.eval(); models.append(m)
print(f'{len(models)} models | {len(widx)} windows | test windows available {len(TEST_STARTS)}')

# the eight players. X channels (28) and the 10 future channels (7 calendar, then u, v, speed) that each player switches off.
GROUPS = {
    'PM2.5 history':        (['pm2.5', 'pm_roll6_mean', 'pm_roll6_std', 'pm_roll24_mean', 'pm_roll24_std', 'pm_delta3'], []),
    'Other pollutants':     (['pm10', 'no2', 'so2', 'co', 'o3'], []),
    'Weather (T, RH)':      (['rh', 'at'], []),
    'Wind (inputs + forecast)': (['u_wind', 'v_wind', 'wspd'], [7, 8, 9]),
    'City context':         (['city_mean_pm', 'city_max_pm', 'pm_minus_city'], []),
    'Calendar (input + future)': (['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'mon_sin', 'mon_cos', 'is_weekend'], [0, 1, 2, 3, 4, 5, 6]),
    'Upwind PM2.5 (transport)': (['adv_pm', 'adv_diff'], []),
    'Wind graph':           ([], []),
}
gnames = list(GROUPS); n_g = len(gnames); GRAPH_G = gnames.index('Wind graph')
gX = [[ADV_NAMES.index(c) for c in GROUPS[g][0]] for g in gnames]; gF = [GROUPS[g][1] for g in gnames]
assert sorted(sum(gX, [])) == list(range(len(ADV_COLS))), 'every input channel must belong to exactly one player'
import itertools, math
SHAP_TERMS = []                                                       # phi_g = sum over coalitions S without g of |S|!(n-|S|-1)!/n! * (v(S+g) - v(S)); S is a bitmask
for g in range(n_g):
    others = [x for x in range(n_g) if x != g]
    for r in range(len(others) + 1):
        for Sset in itertools.combinations(others, r):
            mask = sum(1 << x for x in Sset)
            SHAP_TERMS.append((g, mask, mask | (1 << g), math.factorial(r) * math.factorial(n_g - r - 1) / math.factorial(n_g)))
print('players:', gnames)
print('coalitions:', 2 ** n_g, '| shapley terms:', len(SHAP_TERMS))

5 models | 970 windows | test windows available 2918
players: ['PM2.5 history', 'Other pollutants', 'Weather (T, RH)', 'Wind (inputs + forecast)', 'City context', 'Calendar (input + future)', 'Upwind PM2.5 (transport)', 'Wind graph']
coalitions: 256 | shapley terms: 1024


In [11]:
# main loop: for each chunk of windows run all 256 coalitions through the 5 models; keep the per-seed predictions of the full model
@torch.no_grad()
def run_models(X, At, Afc, Fut, Anc, capture_attn=False):
    outs, attn = [], []
    for m in models:
        store = []
        hk = m.attn.register_forward_hook(lambda mod, inp, out: store.append(out[1].detach().cpu())) if capture_attn else None
        outs.append(m(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), Anc.to(DEVICE)).cpu().numpy())
        if hk is not None: hk.remove(); attn.append(store[0][:, -1, :].numpy())                 # (B*n, 48): last position over the input hours
    return np.stack(outs), (np.mean(attn, axis=0) if capture_attn else None)                     # (5,B,24,n) log preds, (B*n,48)

def apply_mask(mask, X, At, Afc, Fut):
    Xm, Fm, Am, Afm = X.clone(), Fut.clone(), At, Afc
    for g in range(n_g):
        if not (mask >> g) & 1:                                                                     # absent player = training mean (0 after scaling); absent graph = no neighbour exchange
            if gX[g]: Xm[:, :, :, gX[g]] = 0.0
            if gF[g]: Fm[:, :, :, gF[g]] = 0.0
            if g == GRAPH_G: Am, Afm = torch.zeros_like(At), torch.zeros_like(Afc)
    return Xm, Am, Afm, Fm

actual_all = None
CH = 24
phi_all, base_all, fc_all, attn_all, band_all, act_all = [], [], [], [], [], []
t0 = time.time()
for c0 in range(0, len(widx), CH):
    items = [ds[k] for k in widx[c0:c0 + CH]]
    X, At, Afc, Fut, Anc, Y, _w = [torch.stack([it[j] for it in items]) for j in range(7)]
    B = X.size(0)
    v = np.zeros((2 ** n_g, B, N), dtype=np.float32)                                                 # mean 24-hour forecast per coalition
    for mask in range(2 ** n_g):
        Xm, Am, Afm, Fm = apply_mask(mask, X, At, Afc, Fut)
        outs, attn = run_models(Xm, Am, Afm, Fm, Anc, capture_attn=(mask == 2 ** n_g - 1))
        pred = from_log(np.mean(outs, axis=0)); v[mask] = pred.mean(axis=1)                          # ensemble mean in ug/m3, (B,24,n)
        if mask == 2 ** n_g - 1:
            seeds_pred = from_log(outs); full_pred = pred; attn_all.append(attn.reshape(B, N, INPUT_WINDOW))
    phi = np.zeros((B, N, n_g), dtype=np.float32)
    for g, m_without, m_with, w in SHAP_TERMS: phi[:, :, g] += w * (v[m_with] - v[m_without])
    phi_all.append(phi); base_all.append(v[0]); fc_all.append(v[2 ** n_g - 1]); act_all.append(from_log(Y.numpy()))
    band_all.append(np.stack([np.percentile(seeds_pred, 10, axis=0), np.median(seeds_pred, axis=0), np.percentile(seeds_pred, 90, axis=0)]))   # (3,B,24,n)
    if c0 == 0: first_pred = full_pred.copy()
    print(f'  windows {c0 + 1}-{c0 + B} of {len(widx)} | {time.time() - t0:.0f}s')
phi_all, base_all, fc_all, act_all = np.concatenate(phi_all), np.concatenate(base_all), np.concatenate(fc_all), np.concatenate(act_all)
attn_all, band_all = np.concatenate(attn_all), np.concatenate(band_all, axis=1)

# HOW TO CHECK: runs without error and prints one line per chunk of 24 windows; the checks come in the next cell.

  windows 1-24 of 970 | 13s
  windows 25-48 of 970 | 27s
  windows 49-72 of 970 | 40s
  windows 73-96 of 970 | 53s
  windows 97-120 of 970 | 66s
  windows 121-144 of 970 | 79s
  windows 145-168 of 970 | 92s
  windows 169-192 of 970 | 105s
  windows 193-216 of 970 | 117s
  windows 217-240 of 970 | 130s
  windows 241-264 of 970 | 143s
  windows 265-288 of 970 | 156s
  windows 289-312 of 970 | 169s
  windows 313-336 of 970 | 182s
  windows 337-360 of 970 | 195s
  windows 361-384 of 970 | 208s
  windows 385-408 of 970 | 221s
  windows 409-432 of 970 | 234s
  windows 433-456 of 970 | 247s
  windows 457-480 of 970 | 260s
  windows 481-504 of 970 | 273s
  windows 505-528 of 970 | 286s
  windows 529-552 of 970 | 299s
  windows 553-576 of 970 | 312s
  windows 577-600 of 970 | 325s
  windows 601-624 of 970 | 338s
  windows 625-648 of 970 | 350s
  windows 649-672 of 970 | 363s
  windows 673-696 of 970 | 377s
  windows 697-720 of 970 | 390s
  windows 721-744 of 970 | 403s
  windows 745-768 of 970 

In [12]:
# checks, then write the files
saved = np.load(os.path.join(RESULTS_DIR, 'dynadv_test_pred_forecast.npy'))                           # (2918, 24, 37) ug/m3 from notebook 39
# (1) the full-model forecast (mean over the 24 hours) against the predictions saved by notebook 39
saved_mean24 = saved[np.array(widx)].mean(axis=1)                                                     # (B, 37)
d1 = np.abs(fc_all - saved_mean24).max()
print(f'(1) forecast mean vs notebook 39 saved predictions: max abs diff {d1:.4f} ug/m3   (expect about 0.001 or less)')
# (2) alignment with the playback windows: actual PM2.5 of the 24 forecast hours
pb_act = np.array([[[st['actual_24h'][h] for st in f['stations']] for h in range(HORIZON)] for f in PB], dtype=float)
d2 = np.abs(act_all - pb_act).mean()
print(f'(2) actuals vs playback.json actual_24h: mean abs diff {d2:.3f} ug/m3   (expect below 1; a large number means the window indices do not line up)')
resid = fc_all - base_all - phi_all.sum(axis=2)
print(f'(3) Shapley efficiency (forecast - baseline - sum of impacts), max abs: {float(np.abs(resid).max()):.5f}   (expect about 0.001)')

FE = os.path.join(V2_DIR, 'frontend_export_dynadv'); os.makedirs(FE, exist_ok=True)
ew = []
for b, w in enumerate(widx):
    ew.append({'window_idx': w, 'timestamp': PB[b]['timestamp'], 'stations': [
        {'station_id': s, 'baseline': round(float(base_all[b, s]), 1), 'forecast_mean': round(float(fc_all[b, s]), 1),
         'impacts': {gnames[g]: round(float(phi_all[b, s, g]), 1) for g in range(n_g)}, 'attention_1e4': [int(round(x * 10000)) for x in attn_all[b, s]]} for s in range(N)]})
json.dump({'groups': gnames, 'unit': 'ug/m3 effect on the mean 24-hour forecast; impacts + baseline = forecast_mean', 'model': 'WA-TGCN dyn_adv, 5 seeds, real day-old forecast wind',
           'attention': 'weight of the last position on each of the 48 input hours, oldest first, scaled by 1e4; averaged over heads and the 5 seeds',
           'players': 'Wind graph = the hourly wind-driven adjacency (absent: no neighbour exchange); Upwind PM2.5 = the transport features adv_pm and adv_diff',
           'windows': ew}, open(os.path.join(FE, 'explain_windows.json'), 'w'), separators=(',', ':'))
bw = [{'window_idx': w, 'stations': [{'station_id': s, 'p10': [int(round(x)) for x in band_all[0, b, :, s]], 'p50': [int(round(x)) for x in band_all[1, b, :, s]],
                                      'p90': [int(round(x)) for x in band_all[2, b, :, s]]} for s in range(N)]} for b, w in enumerate(widx)]
json.dump({'definition': '10th / 50th / 90th percentile of the 5 seeds (ensemble spread, not a calibrated prediction interval)', 'windows': bw},
          open(os.path.join(FE, 'bands.json'), 'w'), separators=(',', ':'))
mean_imp, abs_imp = phi_all.mean(axis=(0, 1)), np.abs(phi_all).mean(axis=(0, 1))
json.dump({'players': gnames, 'mean_impact_ugm3': {g: float(x) for g, x in zip(gnames, mean_imp)}, 'mean_abs_impact_ugm3': {g: float(x) for g, x in zip(gnames, abs_imp)},
           'share_of_total_abs_impact': {g: float(x / abs_imp.sum()) for g, x in zip(gnames, abs_imp)}, 'baseline_mean': float(base_all.mean()), 'forecast_mean': float(fc_all.mean()),
           'windows': len(widx), 'note': 'computed on the playback windows; impacts are on the MEAN 24 h forecast in ug/m3 (not the RMSE shares of notebook 22)'},
          open(os.path.join(FE, 'explain_global.json'), 'w'), indent=2)
for f in ('explain_windows.json', 'bands.json', 'explain_global.json'): print(f'{f:24} {os.path.getsize(os.path.join(FE, f)) / 1e6:6.1f} MB')
print('\nplayer                         mean impact   mean |impact|   share of total |impact|')
for g, a, b_ in sorted(zip(gnames, mean_imp, abs_imp), key=lambda t: -t[2]): print(f'  {g:28}{a:+9.1f}{b_:14.1f}{100 * b_ / abs_imp.sum():16.1f}%')
print(f'  baseline {base_all.mean():.1f} -> forecast {fc_all.mean():.1f}')
print('band width (p90 - p10), mean over horizon h=+1,+6,+12,+24:', [round(float((band_all[2, :, h, :] - band_all[0, :, h, :]).mean()), 1) for h in (0, 5, 11, 23)])
print('written to', FE)

# HOW TO CHECK: (1) about 0.001, (2) below 1, (3) about 0.001. If (1) or (2) is large, stop and tell me. The impact table should rank PM2.5 history and weather high; the two wind-aware players
# (Upwind PM2.5 and Wind graph) show how much of the forecast the wind-aware design is responsible for - expect small shares for the graph, in line with the ablations.

(1) forecast mean vs notebook 39 saved predictions: max abs diff 0.0068 ug/m3   (expect about 0.001 or less)
(2) actuals vs playback.json actual_24h: mean abs diff 0.009 ug/m3   (expect below 1; a large number means the window indices do not line up)
(3) Shapley efficiency (forecast - baseline - sum of impacts), max abs: 0.00005   (expect about 0.001)
explain_windows.json       17.2 MB
bands.json                 10.8 MB
explain_global.json         0.0 MB

player                         mean impact   mean |impact|   share of total |impact|
  Weather (T, RH)                 +18.4          18.5            25.4%
  PM2.5 history                    +9.5          13.8            19.0%
  Calendar (input + future)        +2.0          11.2            15.3%
  Wind (inputs + forecast)         -3.9           9.6            13.2%
  City context                     +4.3           7.6            10.4%
  Other pollutants                 -3.4           5.1             6.9%
  Upwind PM2.5 (transport)   